# 03 Gold Star Schema
Builds the Gold layer from 'workspace.cms.silver_cms_inpatient': a start schema with 'dim_hospital', 'dim_drg', and a hospital x DRG fact table, plus per-DRG median benchmarks, hospital deviations, and data quality flags (DQ-14, DQ-16, DQ-17). Starts by profiling Silver to confirm the grain and dimension keys. 

In [0]:
-- ===== Profile Silver before building Gold =====
-- Column names and types of the Silver table (29 columns).
DESCRIBE TABLE workspace.cms.silver_cms_inpatient;

In [0]:
-- Silver row count. Result: 145,879, the number the Gold fact table must match.
SELECT COUNT (*) AS n_rows
FROM workspace.cms.silver_cms_inpatient;

In [0]:
-- Grain check: is there exactly one row per hospital x DRG?
-- Result: total rows = distinct (ccn, drg_cd) pairs = 145,879, so yes.
SELECT 
    COUNT (*) AS n_rows,
    COUNT (DISTINCT ccn, drg_cd ) AS n_hospital_drg_pairs

FROM workspace.cms.silver_cms_inpatient; 

In [0]:
-- How many hospitals and DRGs are there? These are the expected row counts for the two dimensions.
-- Result: 2,906 hospitals (ccn) and 540 DRGs (drg_cd).
SELECT 
    COUNT(DISTINCT ccn) AS n_hospitals,
    COUNT(DISTINCT drg_cd) AS n_drgs

FROM workspace.cms.silver_cms_inpatient;

In [0]:
-- Hospital key stability: does any ccn have more than one name, city, state, ZIP, RUCA or address?
-- Inner query counts distinct values per hospital; outer query counts hospitals with any value > 1.
-- Result: 0 unstable hospitals, so ccn is a clean key for gold_dim_hospital.
SELECT COUNT(*) AS n_unstable_hospitals
FROM (
    SELECT
        ccn, 
        COUNT(DISTINCT provider_name) AS n_names,
        COUNT(DISTINCT provider_city) AS n_cities,
        COUNT(DISTINCT state_abbr) AS n_states,
        COUNT(DISTINCT zip5) AS n_zips,
        COUNT(DISTINCT ruca_code) AS n_ruca,
        COUNT(DISTINCT provider_street_addr) AS n_addrs

    FROM workspace.cms.silver_cms_inpatient
    GROUP BY ccn 
    HAVING n_names >1 OR n_cities >1 OR n_states >1 OR n_zips >1 OR n_ruca >1 OR n_addrs > 1

); 

In [0]:
-- Null check on the hospital attributes going into gold_dim_hospital.
-- COUNT(*) minus COUNT(col) = number of nulls. Result: 0 in every column.
SELECT
    COUNT(*) - COUNT(provider_name) AS null_names,
    COUNT(*) - COUNT(provider_city) AS null_cities,
    COUNT(*) - COUNT(state_abbr) AS null_states,
    COUNT(*) - COUNT(zip5) AS null_zips,
    COUNT(*) - COUNT(ruca_code) AS null_ruca

FROM workspace.cms.silver_cms_inpatient;

In [0]:
-- Is drg_cd a clean key? Lists any DRG code with more than one description.
-- Result: no rows, so each drg_cd maps to exactly one drg_desc.
SELECT 
    drg_cd,
    COUNT(*) AS n_desc
FROM workspace.cms.silver_cms_inpatient
GROUP BY drg_cd
HAVING COUNT(DISTINCT drg_desc) > 1; 

In [0]:
-- ===== Week 3: Gold star schema =====
-- DRG dimension: one row per DRG code with its description.
-- DISTINCT works because each drg_cd has only one drg_desc (checked in cell 8).
CREATE OR REPLACE TABLE workspace.cms.gold_dim_drg AS 
SELECT DISTINCT 
    drg_cd,
    drg_desc

FROM workspace.cms.silver_cms_inpatient;

In [0]:
-- Check gold_dim_drg: expect 540 rows, matching the distinct drg_cd count in cell 5.
SELECT COUNT(*) 
FROM workspace.cms.gold_dim_drg;

In [0]:
-- Hospital dimension: one row per hospital (ccn) with its descriptive attributes.
-- DISTINCT works because each ccn has exactly one value per attribute (checked in cell 6).
-- Check: 2,906 rows, matching the distinct ccn count in cell 5.
CREATE OR REPLACE TABLE workspace.cms.gold_dim_hospital AS 
SELECT DISTINCT 
    ccn,
    provider_name,
    provider_city,
    state_abbr,
    zip5,
    ruca_code,
    provider_street_addr,
    ruca_desc,
    urban_rural,
    is_maryland

FROM workspace.cms.silver_cms_inpatient;

SELECT COUNT (*) 
FROM workspace.cms.gold_dim_hospital;

In [0]:
-- Fact table: one row per hospital x DRG (same grain as Silver) with the measures only.
-- ccn joins to gold_dim_hospital, drg_cd joins to gold_dim_drg.
-- Check: 145,879 rows, matching Silver exactly.
CREATE OR REPLACE TABLE workspace.cms.gold_fact_inpatient AS
SELECT 
ccn,
drg_cd,
tot_discharges,
avg_submitted_charge, 
avg_total_payment,
avg_medicare_payment,
tot_submitted_charge,
tot_total_payment,
tot_medicare_payment,
markup_ratio

FROM workspace.cms.silver_cms_inpatient;

SELECT COUNT(*)
FROM workspace.cms.gold_fact_inpatient;

In [0]:
-- DRG benchmark: one row per DRG with the median avg_total_payment across hospitals.
-- Plain median (each hospital counts once); Maryland excluded via the hospital dim (DQ-13).
-- is_thin_drg (DQ-16): fewer than 30 hospitals, so the median is less stable.
-- Checks: 539 rows (DRG 950 drops out, see DQ-22); 229 DRGs flagged thin.
CREATE OR REPLACE TABLE workspace.cms.gold_drg_benchmark AS
SELECT
    drg_cd,
    COUNT (*) AS n_hospitals,
    median(avg_total_payment) AS median_payment,
    percentile(avg_total_payment, 0.25) AS q1_payment,
    percentile(avg_total_payment, 0.75) AS q3_payment,

COUNT(*) < 30 AS is_thin_drg

FROM workspace.cms.gold_fact_inpatient f
JOIN workspace.cms.gold_dim_hospital h
ON f.ccn = h.ccn
WHERE h.is_maryland = false
GROUP BY drg_cd;

SELECT 
    COUNT(*)
FROM workspace.cms.gold_drg_benchmark;

SELECT COUNT (*) FROM workspace.cms.gold_drg_benchmark 
WHERE is_thin_drg;

In [0]:
-- DQ-22: find DRGs that are in gold_dim_drg but missing from gold_drg_benchmark.
-- LEFT JOIN keeps every DRG; IS NULL keeps only those with no benchmark row.
-- Result: DRG 950 (Aftercare without CC/MCC), reported only by Maryland hospitals.
SELECT 
    d.drg_cd, 
    d.drg_desc
FROM workspace.cms.gold_dim_drg d
LEFT JOIN workspace.cms.gold_drg_benchmark b
ON d.drg_cd = b.drg_cd
WHERE b.drg_cd IS NULL;

In [0]:
-- DQ-17: hospitals with few DRGs. One row per hospital (ccn) with its DRG count;
-- is_low_drg_hospital is true when a hospital reports fewer than 5 DRGs.
-- Checks: 2,906 rows (one per hospital); 370 hospitals flagged.
CREATE OR REPLACE TABLE workspace.cms.gold_hospital_drg_count AS
SELECT
    ccn,
    COUNT (*) AS n_drgs, 
    COUNT (*) < 5 is_low_drg_hospital

FROM 
    workspace.cms.gold_fact_inpatient

GROUP BY ccn;

SELECT COUNT (*) FROM workspace.cms.gold_hopspital_drg_count;

SELECT COUNT (*) FROM workspace.cms.gold_hopspital_drg_count WHERE is_low_drg_hospital;

In [0]:
-- Hospital deviation: one row per hospital x DRG, comparing each payment to its DRG benchmark.
-- payment_ratio = hospital payment / DRG median (1.25 = 25% above typical). Maryland rows are scored too.
-- is_payment_outlier (DQ-14): payment above Q3 + 3 x IQR within the DRG, only in non-thin DRGs.
-- Checks: 145,878 rows (DRG 950 has no benchmark, see DQ-22); 2,358 rows flagged as outliers.
CREATE OR REPLACE TABLE workspace.cms.gold_hospital_drg_deviation AS

SELECT 
    ccn,
    f.drg_cd, 
    avg_total_payment,
    median_payment,
    avg_total_payment / median_payment AS payment_ratio,
    b.q1_payment, 
    b.q3_payment,
    b.is_thin_drg,
    avg_total_payment > (b.q3_payment + 3 * (b.q3_payment - b.q1_payment)) AND NOT b.is_thin_drg AS is_payment_outlier
    

FROM 
    workspace.cms.gold_fact_inpatient f

JOIN 
    workspace.cms.gold_drg_benchmark b
ON 
    f.drg_cd = b.drg_cd;

SELECT COUNT (*) FROM workspace.cms.gold_hospital_drg_deviation;

SELECT COUNT (*) FROM workspace.cms.gold_hospital_drg_deviation WHERE is_payment_outlier;

In [0]:
SELECT 
    COUNT (*) FROM workspace.cms.gold_hospital_drg_deviation;

In [0]:
SELECT median(payment_ratio) FROM workspace.cms.gold_hospital_drg_deviation; 